In [0]:
# Read active streaming configurations

control_df = spark.sql("""
    SELECT *
    FROM control_table
    WHERE is_active = true
      AND environment = 'DEV'
      AND is_streaming = true
""")

control_df.show(truncate=False)

In [0]:
# Select the Kraken configuration

kraken_config = (
    control_df
    .filter(control_df.config_id == 1)
    .first()
)

print("Source Name:", kraken_config.source_name)
print("Source Type:", kraken_config.source_type)
print("Source Entity:", kraken_config.source_entity)
print("Target Table:", kraken_config.target_table)
print("Load Type:", kraken_config.load_type)

In [0]:
import websocket
import json

source_entity = kraken_config.source_entity

ws = websocket.create_connection(
    "wss://ws.kraken.com/v2"
)

subscribe_message = {
    "method": "subscribe",
    "params": {
        "channel": "trade",
        "symbol": [source_entity]
    }
}

ws.send(json.dumps(subscribe_message))

print("Connected to:", source_entity)

In [0]:
# Receive trade messages from Kraken

trade_rows = []

while len(trade_rows) < 20:

    message = ws.recv()
    data = json.loads(message)

    if data.get("channel") == "trade":

        trades = data.get("data", [])

        for trade in trades:

            trade_rows.append((
                trade["symbol"],
                trade["side"],
                float(trade["price"]),
                float(trade["qty"]),
                int(trade["trade_id"]),
                trade["ord_type"],
                trade["timestamp"]
            ))

            if len(trade_rows) >= 20:
                break

print("Trades collected:", len(trade_rows))

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    LongType,
    TimestampType
)

bronze_schema = StructType([
    StructField("symbol", StringType(), False),
    StructField("side", StringType(), False),
    StructField("price", DoubleType(), False),
    StructField("qty", DoubleType(), False),
    StructField("trade_id", LongType(), False),
    StructField("ord_type", StringType(), False),
    StructField("event_timestamp", TimestampType(), False)
])

print("Bronze schema created")

In [0]:
from datetime import datetime

trade_rows_fixed = []

for row in trade_rows:
    trade_rows_fixed.append((
        row[0],
        row[1],
        row[2],
        row[3],
        row[4],
        row[5],
        datetime.fromisoformat(
            row[6].replace("Z", "+00:00")
        )
    ))

bronze_df = spark.createDataFrame(
    trade_rows_fixed,
    bronze_schema
)

bronze_df.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp, lit

bronze_df_enriched = (
    bronze_df
    .withColumn("source_system", lit(kraken_config.source_name))
    .withColumn("ingestion_timestamp", current_timestamp())
)

bronze_df_enriched.printSchema()
bronze_df_enriched.show(5, truncate=False)

In [0]:
target_table = kraken_config.target_table

bronze_df_enriched.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(target_table)

print("Data written to:", target_table)

In [0]:
%sql
SELECT *
FROM bronze_crypto_trades
ORDER BY ingestion_timestamp DESC, event_timestamp DESC;

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM bronze_crypto_trades;

In [0]:
%sql
SELECT
    trade_id,
    COUNT(*) AS record_count
FROM bronze_crypto_trades
GROUP BY trade_id
HAVING COUNT(*) > 1
ORDER BY record_count DESC;

In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT trade_id) AS unique_trade_ids,
    COUNT(*) - COUNT(DISTINCT trade_id) AS duplicate_records,
    COUNT(*) FILTER (WHERE price <= 0) AS invalid_price_records,
    COUNT(*) FILTER (WHERE qty <= 0) AS invalid_quantity_records,
    COUNT(*) FILTER (WHERE event_timestamp IS NULL) AS null_timestamp_records
FROM bronze_crypto_trades;

In [0]:
# Read retry configuration from control table

max_retries = kraken_config.max_retries
retry_interval_seconds = kraken_config.retry_interval_seconds

print("Maximum retries:", max_retries)
print("Retry interval:", retry_interval_seconds, "seconds")

In [0]:
import time

def connect_to_kraken(symbol, max_retries, retry_interval_seconds):
    for attempt in range(1, max_retries + 1):

        try:
            ws = websocket.create_connection(
                "wss://ws.kraken.com/v2",
                timeout=10
            )

            subscribe_message = {
                "method": "subscribe",
                "params": {
                    "channel": "trade",
                    "symbol": [symbol]
                }
            }

            ws.send(json.dumps(subscribe_message))

            print(
                f"Connected to {symbol} "
                f"(attempt {attempt})"
            )

            return ws

        except Exception as e:

            print(
                f"Connection failed "
                f"(attempt {attempt}/{max_retries}): {e}"
            )

            if attempt < max_retries:
                print(
                    f"Retrying in "
                    f"{retry_interval_seconds} seconds..."
                )
                time.sleep(retry_interval_seconds)

    raise RuntimeError(
        f"Unable to connect to Kraken after "
        f"{max_retries} attempts"
    )

In [0]:
ws = connect_to_kraken(
    kraken_config.source_entity,
    max_retries,
    retry_interval_seconds
)

In [0]:
# Reload the configuration after adding source_endpoint

control_df = spark.sql("""
    SELECT *
    FROM control_table
    WHERE is_active = true
      AND environment = 'DEV'
      AND is_streaming = true
""")

kraken_config = (
    control_df
    .filter(control_df.config_id == 1)
    .first()
)

source_endpoint = kraken_config.source_endpoint

print("Source Endpoint:", source_endpoint)

In [0]:
def connect_to_source(
    source_endpoint,
    source_entity,
    max_retries,
    retry_interval_seconds
):
    for attempt in range(1, max_retries + 1):

        try:
            ws = websocket.create_connection(
                source_endpoint,
                timeout=10
            )

            subscribe_message = {
                "method": "subscribe",
                "params": {
                    "channel": "trade",
                    "symbol": [source_entity]
                }
            }

            ws.send(json.dumps(subscribe_message))

            print(
                f"Connected to {source_entity} "
                f"(attempt {attempt})"
            )

            return ws

        except Exception as e:

            print(
                f"Connection failed "
                f"(attempt {attempt}/{max_retries}): {e}"
            )

            if attempt < max_retries:
                print(
                    f"Retrying in "
                    f"{retry_interval_seconds} seconds..."
                )
                time.sleep(retry_interval_seconds)

    raise RuntimeError(
        f"Unable to connect after "
        f"{max_retries} attempts"
    )

In [0]:
ws = connect_to_source(
    source_endpoint,
    kraken_config.source_entity,
    kraken_config.max_retries,
    kraken_config.retry_interval_seconds
)